# Processamento de Sinais II — Lista 1
**Aluno:** Pedro Batsos  
**Período:** 2026.2

Solução computacional das 14 questões. Execute as células em ordem. O notebook procura `dataset.zip` na pasta do notebook e extrai as imagens automaticamente.

In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage import color, exposure, filters, morphology, transform, util

HERE = Path.cwd()
REPO = HERE if (HERE / 'data').exists() else HERE.parent
DATA_ROOT = REPO / 'data'
ZIP = DATA_ROOT / 'dataset.zip'
if ZIP.exists() and not (DATA_ROOT / 'dataset').exists():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA_ROOT)
DATA = DATA_ROOT / 'dataset'

def load(name, gray=False):
    p = DATA / name
    a = np.asarray(Image.open(p))
    if gray and a.ndim == 3:
        a = color.rgb2gray(a)
    return a

def gray8(a):
    a = np.asarray(a)
    if a.ndim == 3: a = color.rgb2gray(a)
    if a.dtype.kind in 'ui': return a.astype(np.float32) / np.iinfo(a.dtype).max
    return np.clip(a.astype(float), 0, 1)

def show(items, cols=4, cmap='gray', figsize=None):
    rows = int(np.ceil(len(items) / cols)); figsize = figsize or (4*cols, 4*rows)
    fig, ax = plt.subplots(rows, cols, figsize=figsize, squeeze=False); ax = ax.ravel()
    for i, (title, img) in enumerate(items):
        ax[i].imshow(img, cmap=cmap, vmin=0 if np.asarray(img).ndim == 2 else None, vmax=1 if np.asarray(img).ndim == 2 else None)
        ax[i].set_title(title); ax[i].axis('off')
    for a in ax[len(items):]: a.axis('off')
    plt.tight_layout(); plt.show()

lena_gray = gray8(load('lena2.tif', gray=True))
lena_color = load('lena3.tif')
frog = gray8(load('frog.tif', gray=True))
brain = gray8(load('brain.jpg', gray=True))
print('Dataset:', DATA.resolve())

## 1) Redução da resolução da Lena gray
A interpolação `area` representa a média dos pixels agrupados. A exibição usa o mesmo tamanho da imagem original para permitir comparação visual.

In [ ]:
reduced = [transform.resize(lena_gray, (n, n), order=1, anti_aliasing=True) for n in (256, 128, 64)]
show([('Original 512×512', lena_gray), *[(f'{n}×{n}', im) for n, im in zip((256,128,64), reduced)]], cols=4)
print('A redução provoca perda progressiva de detalhes finos e suavização; a interpolação evita aliasing excessivo.')

## 2) Canais RGB da Lena color
Cada canal é uma matriz escalar: o brilho mostrado em cada pixel indica a intensidade daquela componente, por isso a visualização monocromática é natural.

In [ ]:
show([('RGB', lena_color), ('R', lena_color[...,0]), ('G', lena_color[...,1]), ('B', lena_color[...,2])], cols=4, cmap='gray')

## 3) Bits por pixel e quantização
A Lena color RGB possui 24 bpp (8 bits por canal); a Lena gray possui 8 bpp. A quantização reduz os níveis disponíveis para 2^b, produzindo contornos falsos e banding quando há poucos bits.

In [ ]:
def quantize(im, bits):
    levels = 2**bits
    return np.floor(im*(levels-1)+0.5)/(levels-1)
show([('Original — 8 bpp', lena_gray), *[(f'{b} bpp ({2**b} níveis)', quantize(lena_gray,b)) for b in (6,4,2,1)]], cols=5)

## 4) Quadrado central e percepção de brilho
O limiar de percepção depende do fundo e da adaptação visual. As duas condições são simétricas numericamente, mas a percepção não precisa ser; a resposta humana ao brilho é aproximadamente logarítmica, isto é, diferenças relativas de luminância tendem a ser percebidas de forma mais uniforme que diferenças absolutas.

In [ ]:
base = np.full((256,256), 128/255, float); s=slice(112,144)
levels=np.array([0,32,64,96,128,160,192,224,255])/255
fig, ax=plt.subplots(2,4,figsize=(12,6))
for j,v in enumerate(levels[:4]):
    im=base.copy(); im[s,s]=v; ax[0,j].imshow(im,cmap='gray',vmin=0,vmax=1); ax[0,j].set_title(f'escuro: {round(v*255)}'); ax[0,j].axis('off')
for j,v in enumerate(levels[5:]):
    im=base.copy(); im[s,s]=v; ax[1,j].imshow(im,cmap='gray',vmin=0,vmax=1); ax[1,j].set_title(f'claro: {round(v*255)}'); ax[1,j].axis('off')
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()
print('Próximo de 128, o quadrado tende a desaparecer; a adaptação ao fundo explica a ilusão de uniformidade.')

## 5) Ajuste de brilho (escala 1,2)


In [ ]:
bright=np.clip(frog*1.2,0,1); show([('Frog original',frog),('Brilho ×1,2',bright)],cols=2); print('O escalonamento aumenta a luminância; pixels claros que ultrapassam 1 são saturados.')

## 6) Ajuste de contraste por gamma = 1,5
A transformação s = r^gamma escurece valores normalizados abaixo de 1 e comprime a faixa de tons claros; a escolha da convenção gamma deve ser explicitada.

In [ ]:
gamma=1.5; gamma_img=frog**gamma; show([('Frog original',frog),(r'$s=r^{1.5}$',gamma_img)],cols=2); print('Com gamma > 1, tons médios ficam mais escuros nesta convenção.')

## 7) Supressão de ruído por média
A média de 32 observações independentes reduz o desvio-padrão do ruído aproximadamente por sqrt(32), preservando a estrutura comum à imagem.

In [ ]:
rng=np.random.default_rng(7); sigma=.12; noisy=np.clip(brain+rng.normal(0,sigma,(32,*brain.shape)),0,1); mean_img=noisy.mean(axis=0); show([('Brain original',brain),('Uma imagem ruidosa',noisy[0]),('Média de 32',mean_img)],cols=3); print(f'RMSE da média: {np.sqrt(np.mean((mean_img-brain)**2)):.4f}; sigma teórico após média: {sigma/np.sqrt(32):.4f}')

## 8–10) Histogramas, equalização global e adaptativa
Os histogramas revelam distribuição de tons e contraste: imagens concentradas em uma faixa têm contraste reduzido; distribuições amplas ocupam melhor a faixa dinâmica. A equalização global redistribui a CDF; a CLAHE/adaptativa atua localmente e pode realçar detalhes, mas também o ruído.

In [ ]:
imgs={'Lena gray':lena_gray,'Moon':gray8(load('moon.jpg')),'Bay':gray8(load('bay.jpg')),'Brain':brain}
fig,ax=plt.subplots(2,4,figsize=(16,7))
for j,(name,im) in enumerate(imgs.items()): ax[0,j].imshow(im,cmap='gray',vmin=0,vmax=1); ax[0,j].set_title(name); ax[0,j].axis('off'); ax[1,j].hist(im.ravel(),bins=256,range=(0,1),color='black'); ax[1,j].set_xlim(0,1); ax[1,j].set_title('Histograma')
plt.tight_layout(); plt.show()
equalized={k:exposure.equalize_hist(v) for k,v in imgs.items()}
for block in (8,16):
    adapted={k:exposure.equalize_adapthist(v, kernel_size=(block,block), clip_limit=.03) for k,v in imgs.items()}
    show([(f'{k} — global', equalized[k]) for k in imgs]+[(f'{k} — adaptativa {block}×{block}', adapted[k]) for k in imgs],cols=4,figsize=(16,16))
print('A janela 8×8 realça detalhes menores e tende a gerar mais ruído; 16×16 é mais estável. A melhor escolha depende da escala dos detalhes.')

## 11–12) Canais e conversão da Balloons
Os canais RGB são intensidades monocromáticas. A média simples atribui pesos iguais; a conversão padrão usa luminância perceptual (aproximadamente 0,2126R + 0,7152G + 0,0722B), preservando melhor a sensibilidade do olho ao verde.

In [ ]:
ball=load('balloons.tif'); ballf=ball[...,:3]/(255 if ball.dtype.kind in 'ui' else 1);
mean_gray=ballf.mean(axis=2); standard=color.rgb2gray(ballf)
show([('Balloons',ballf),('R',ballf[...,0]),('G',ballf[...,1]),('B',ballf[...,2])],cols=4,cmap='gray')
show([('Média RGB',mean_gray),('Luminância padrão',standard)],cols=2); print('As imagens diferem porque a média trata R, G e B igualmente, enquanto a luminância pondera o verde mais fortemente.')

## 13) Segmentação da cor vermelha em Peppers
A limiarização em níveis de cinza não separa perfeitamente matiz e brilho: sombras, reflexos, texturas e regiões com luminância semelhante ao vermelho entram ou saem da máscara. Por isso aparecem imperfeições.

In [ ]:
pep=load('peppers3.tif'); pepf=pep[...,:3]/(255 if pep.dtype.kind in 'ui' else 1); pgray=color.rgb2gray(pepf); mask=(pepf[...,0]>0.45)&(pepf[...,0]>pepf[...,1]*1.25)&(pepf[...,0]>pepf[...,2]*1.25); masked=pepf*mask[...,None]; show([('Original',pepf),('Cinza',pgray),('Máscara vermelha',mask),('Aplicada',masked)],cols=4,cmap='gray')

## 14) Otsu global e limiarização local na Guia
Otsu escolhe um único limiar maximizando a separação entre duas classes. Em iluminação não uniforme, a limiarização adaptativa calcula o limiar em janelas; a variância local permite evitar decisões em regiões quase uniformes. A comparação abaixo testa janelas e imprime o melhor compromisso visual.

In [ ]:
guia=gray8(load('guia.jpg')); otsu=guia>filters.threshold_otsu(guia)
windows=(15,31,51,81); local=[]
for w in windows:
    t=filters.threshold_local(guia,w,method='gaussian',offset=0.02); local.append(guia>t)
show([('Guia',guia),('Otsu global',otsu),*[(f'Local — janela {w}',m) for w,m in zip(windows,local)]],cols=3,figsize=(14,10))
print('Em geral, uma janela intermediária (31×31 ou 51×51) equilibra detalhe e estabilidade; confirme visualmente no seu ambiente, pois o resultado depende da iluminação da imagem.')